In [2]:
import joblib
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import MACCSkeys
from rdkit import DataStructs
from mordred import Calculator, descriptors
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
import os

TRAIN_MACCS_PATH   = "MACCS_c+g_con_actividad.csv"
TRAIN_MORDRED_PATH = "Merge_ChEMBL_gallinamides.csv"
TRAIN_SMILES_COL   = "smiles_std"
OUTPUT_DIR_VS      = "cribado virtual"
os.makedirs(OUTPUT_DIR_VS, exist_ok=True)

In [3]:
df_raw = pd.read_csv("Plasmodium_all_compounds.csv", sep=';')
df_filt = df_raw.copy()

lfc        = rdMolStandardize.LargestFragmentChooser()   # se queda con el fragmento principal (quita sales)
uncharger  = rdMolStandardize.Uncharger()                 # neutraliza cargas donde es posible
normalizer = rdMolStandardize.Normalizer()
 
def standardize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        mol = lfc.choose(mol)
        mol = normalizer.normalize(mol)
        mol = uncharger.uncharge(mol)
        Chem.SanitizeMol(mol)
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None
 
# =============================================================================
# Limpieza completa: estandarización + armonización de unidades a uM + Label
# =============================================================================
steps_log = []
 
def get_full_cleaned_dataset(df_raw):
    df = df_raw.copy()
 
    # SMILES -> SMILES canonicos (estandarizados)
    df['canonical_smiles'] = df['Smiles'].apply(standardize_smiles)
    n_before = len(df)
    df = df.dropna(subset=['canonical_smiles'])
    steps_log.append(('Canonicalize SMILES (drop invalid)', n_before, len(df)))
 
    df['Standard Value'] = pd.to_numeric(df['Standard Value'], errors='coerce')
    n_before = len(df)
    df = df.dropna(subset=['Standard Value', 'Standard Units', 'canonical_smiles'])
    steps_log.append(('Drop rows without valid activity value/units', n_before, len(df)))
 
    def convert_to_um(row):
        val = row['Standard Value']
        unit = str(row['Standard Units']).lower()
        smiles = row['canonical_smiles']
        if unit == 'nm':
            return val / 1000
        if 'um' in unit or 'µm' in unit:
            return val
        if 'ug.ml-1' in unit or 'ug/ml' in unit:
            mw = row['Molecular Weight'] if 'Molecular Weight' in row and pd.notna(row['Molecular Weight']) else None
            if not mw or mw <= 0:
                mol = Chem.MolFromSmiles(smiles)
                mw = Descriptors.MolWt(mol) if mol else None
            if mw:
                return (val * 1000.0) / mw
        return None
 
    df['IC50_uM'] = df.apply(convert_to_um, axis=1)
    n_before = len(df)
    df = df.dropna(subset=['IC50_uM'])
    steps_log.append(('Convert activity to uM (drop unconvertible units)', n_before, len(df)))
 
    # Un valor por ChEMBL ID (mediana si hay varias medidas)
    df_train = df.groupby('Molecule ChEMBL ID').agg(
        smiles_std=('canonical_smiles', 'first'),
        activity_uM=('IC50_uM', 'median'),
        n_measurements=('IC50_uM', 'count')
    ).reset_index()
 
    # eliminar duplicados: mismo compuesto (mismo SMILES canonico) bajo distintos ChEMBL ID
    n_before_dedup = len(df_train)
    df_train = df_train.groupby('smiles_std').agg(
        **{
            'Molecule ChEMBL ID': ('Molecule ChEMBL ID', 'first'),
            'activity_uM': ('activity_uM', 'median'),
            'n_measurements': ('n_measurements', 'sum'),
        }
    ).reset_index()
    steps_log.append(('Deduplicate by canonical SMILES', n_before_dedup, len(df_train)))
 
    df_train['Label'] = (df_train['activity_uM'] < 5).astype(int)
 
    return df_train
 
 
# =============================================================================
# Ejecución
# =============================================================================
df_raw = pd.read_csv("Plasmodium_all_compounds.csv", sep=';')
print(f"Filas leídas del export de ChEMBL: {len(df_raw)}")
 
df_candidatos = get_full_cleaned_dataset(df_raw)
 
print("\nResumen de limpieza:")
for step_name, n_before, n_after in steps_log:
    print(f"  - {step_name}: {n_before} -> {n_after}")
print(f"\nCompuestos candidatos (SMILES + actividad armonizada, antes de excluir train): {len(df_candidatos)}")
 
# =============================================================================
# Exclusión (obligatoria) de compuestos ya usados en el entrenamiento de FP2
# =============================================================================
train_smiles = set()
for path in [TRAIN_MACCS_PATH, TRAIN_MORDRED_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"No encuentro '{path}'. La exclusión de compuestos de entrenamiento es obligatoria: "
            f"copia aquí los CSV de entrenamiento originales o corrige la ruta en la CONFIGURACIÓN "
            f"(TRAIN_MACCS_PATH / TRAIN_MORDRED_PATH)."
        )
    df_tr = pd.read_csv(path)
    assert TRAIN_SMILES_COL in df_tr.columns, (
        f"'{path}' no tiene la columna '{TRAIN_SMILES_COL}'. Ajusta TRAIN_SMILES_COL en la CONFIGURACIÓN."
    )
    train_smiles.update(df_tr[TRAIN_SMILES_COL].dropna().unique().tolist())
 
n_before = len(df_candidatos)
df_candidatos = df_candidatos[~df_candidatos['smiles_std'].isin(train_smiles)].reset_index(drop=True)
print(f"\nExcluidos por ya estar en el training set de FP2: {n_before - len(df_candidatos)}")
print(f"Candidatos finales para el cribado: {len(df_candidatos)}")
 
df_candidatos.head()

Filas leídas del export de ChEMBL: 66597

Resumen de limpieza:
  - Canonicalize SMILES (drop invalid): 66597 -> 66477
  - Drop rows without valid activity value/units: 66477 -> 64133
  - Convert activity to uM (drop unconvertible units): 64133 -> 64039
  - Deduplicate by canonical SMILES: 30867 -> 30575

Compuestos candidatos (SMILES + actividad armonizada, antes de excluir train): 30575

Excluidos por ya estar en el training set de FP2: 445
Candidatos finales para el cribado: 30130


,smiles_std,Molecule ChEMBL ID,activity_uM,n_measurements,Label
0,Br/C=C/c1ccc2ccccc2n1,CHEMBL120356,32.000,1,0
1,BrC(COc1cccc2c[n+](Cc3ccccc3)ccc12)CN(Cc1ccccc...,CHEMBL3221798,0.695,2,1
2,BrC1=NOC(c2nc(-c3ccc4c(c3)OCCO4)no2)C1,CHEMBL4868589,31.250,2,0
3,BrC1=NOC(c2nc(-c3ccc4ccccc4c3)no2)C1,CHEMBL4869503,27.500,2,0
4,BrC1=NOC(c2nc(-c3ccccc3)cs2)C1,CHEMBL4871929,42.750,2,0


In [8]:
maccs_bundle = joblib.load("maccs_bundle.joblib")
mordred_bundle = joblib.load("mordred_bundle.joblib")

for name, bundle in [("MACCS", maccs_bundle), ("Mordred", mordred_bundle)]:
    print(f"--- {name} ---")
    print(f"  n_features      : {len(bundle['feature_columns'])}")
    print(f"  descriptor_type : {bundle['descriptor_type']}")
    print(f"  modelos top-3   : {list(bundle['top3_models'].keys())}")
    print(f"  AD threshold    : {bundle['ad_threshold']:.4f}  (k={bundle['ad_n_neighbors']})")
    print(f"  label_meaning   : {bundle['label_meaning']}")
    print()

--- MACCS ---
  n_features      : 167
  descriptor_type : maccs
  modelos top-3   : ['SVM_RBF', 'AdaBoost', 'SVM_Linear']
  AD threshold    : 13.5531  (k=5)
  label_meaning   : {'1': 'Activo (IC50 < 5 uM frente a Falcipaina-2)', '0': 'Inactivo (IC50 >= 5 uM frente a Falcipaina-2)'}

--- Mordred ---
  n_features      : 317
  descriptor_type : mordred
  modelos top-3   : ['KNeighbors', 'GradientBoosting', 'ExtraTrees']
  AD threshold    : 14.9951  (k=5)
  label_meaning   : {'1': 'Activo (IC50 < 5 uM frente a Falcipaina-2)', '0': 'Inactivo (IC50 >= 5 uM frente a Falcipaina-2)'}



In [9]:
from rdkit.Chem import MACCSkeys
from rdkit import DataStructs

def compute_maccs_df(smiles_list):
    rows, valid_idx = [], []
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        fp = MACCSkeys.GenMACCSKeys(mol)
        arr = np.zeros((167,), dtype=int)
        DataStructs.ConvertToNumpyArray(fp, arr)
        rows.append(arr)
        valid_idx.append(i)
    cols = [f"MACCS_{i}" for i in range(167)]
    X = pd.DataFrame(rows, columns=cols)
    return X, valid_idx

print("Calculando MACCS keys para los candidatos...")
X_maccs_raw, valid_idx_maccs = compute_maccs_df(df_candidatos["smiles_std"].tolist())
df_maccs_cand = df_candidatos.iloc[valid_idx_maccs].reset_index(drop=True)
X_maccs = X_maccs_raw.reindex(columns=maccs_bundle["feature_columns"])

mask_completo = X_maccs.notna().all(axis=1)
n_incompletos = int((~mask_completo).sum())
if n_incompletos:
    print(f"  - descartadas por features MACCS incompletas (sin imputar): {n_incompletos}")

X_maccs = X_maccs[mask_completo].reset_index(drop=True)
df_maccs_cand = df_maccs_cand[mask_completo].reset_index(drop=True)

print(f"Matriz MACCS candidatos: {X_maccs.shape}")

Calculando MACCS keys para los candidatos...
Matriz MACCS candidatos: (30130, 167)


In [10]:
from mordred import Calculator, descriptors

mols_mordred, valid_idx_mordred = [], []
for i, smi in enumerate(df_candidatos["smiles_std"]):
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        mols_mordred.append(mol)
        valid_idx_mordred.append(i)

df_mordred_cand = df_candidatos.iloc[valid_idx_mordred].reset_index(drop=True)

print(f"Calculando descriptores Mordred 2D para {len(mols_mordred)} moléculas...")
calc = Calculator(descriptors, ignore_3D=True)
df_desc_raw = calc.pandas(mols_mordred)

X_mordred = df_desc_raw.apply(pd.to_numeric, errors="coerce")
X_mordred = X_mordred.reindex(columns=mordred_bundle["feature_columns"])

mask_completo = X_mordred.notna().all(axis=1)
n_incompletos = int((~mask_completo).sum())
if n_incompletos:
    print(f"  - descartadas por descriptores Mordred incompletos (sin imputar): {n_incompletos}")

X_mordred = X_mordred[mask_completo].reset_index(drop=True)
df_mordred_cand = df_mordred_cand[mask_completo].reset_index(drop=True)

print(f"Matriz Mordred candidatos: {X_mordred.shape}")

Calculando descriptores Mordred 2D para 30130 moléculas...


  0%|          | 82/30130 [00:04<36:28, 13.73it/s]  

c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


  1%|          | 176/30130 [00:06<12:11, 40.93it/s]

c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


...

c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


  3%|▎         | 928/30130 [00:48<22:03, 22.06it/s]

c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


100%|██████████| 30130/30130 [19:16<00:00, 26.05it/s]  


  - descartadas por descriptores Mordred incompletos (sin imputar): 378
Matriz Mordred candidatos: (29752, 317)


In [11]:
def apply_ad(X, bundle):
    X_scaled = bundle["ad_scaler"].transform(X)
    dist, _ = bundle["ad_nn"].kneighbors(X_scaled, n_neighbors=bundle["ad_n_neighbors"])
    mean_dist = dist.mean(axis=1)
    in_ad = mean_dist < bundle["ad_threshold"]
    return mean_dist, in_ad

dist_maccs, in_ad_maccs = apply_ad(X_maccs, maccs_bundle)
dist_mordred, in_ad_mordred = apply_ad(X_mordred, mordred_bundle)

df_maccs_cand["ad_dist"] = dist_maccs
df_maccs_cand["in_AD_maccs"] = in_ad_maccs

df_mordred_cand["ad_dist"] = dist_mordred
df_mordred_cand["in_AD_mordred"] = in_ad_mordred

print(f"MACCS   - dentro del AD: {in_ad_maccs.sum()} / {len(in_ad_maccs)} ({in_ad_maccs.mean():.1%})")
print(f"Mordred - dentro del AD: {in_ad_mordred.sum()} / {len(in_ad_mordred)} ({in_ad_mordred.mean():.1%})")

MACCS   - dentro del AD: 22822 / 30130 (75.7%)
Mordred - dentro del AD: 16356 / 29752 (55.0%)


In [12]:
def predict_bundle(X, bundle):
    """Devuelve prob/pred del top-1 y el nº de votos 'activo' entre los top-3."""
    ranking = bundle["top3_ranking"]     
    models  = bundle["top3_models"]

    top1_name = ranking[0]["model"]
    top1_model = models[top1_name]
    if hasattr(top1_model, "predict_proba"):
        prob_top1 = top1_model.predict_proba(X)[:, 1]
    else:
        prob_top1 = top1_model.decision_function(X)
    pred_top1 = (prob_top1 >= 0.5).astype(int)

    votes = np.zeros(len(X), dtype=int)
    for rank_info in ranking:
        m = models[rank_info["model"]]
        if hasattr(m, "predict_proba"):
            p = m.predict_proba(X)[:, 1]
        else:
            p = m.decision_function(X)
        votes += (p >= 0.5).astype(int)

    return prob_top1, pred_top1, votes, top1_name

prob_maccs, pred_maccs, votes_maccs, name_maccs = predict_bundle(X_maccs, maccs_bundle)
prob_mordred, pred_mordred, votes_mordred, name_mordred = predict_bundle(X_mordred, mordred_bundle)

df_maccs_cand["maccs_model_top1"]   = name_maccs
df_maccs_cand["maccs_prob_top1"]    = prob_maccs
df_maccs_cand["maccs_pred_top1"]    = pred_maccs
df_maccs_cand["maccs_top3_votes"]   = votes_maccs   

df_mordred_cand["mordred_model_top1"] = name_mordred
df_mordred_cand["mordred_prob_top1"]  = prob_mordred
df_mordred_cand["mordred_pred_top1"]  = pred_mordred
df_mordred_cand["mordred_top3_votes"] = votes_mordred

print(f"MACCS   top-1 = {name_maccs}   | predichos activos: {pred_maccs.sum()} / {len(pred_maccs)}")
print(f"Mordred top-1 = {name_mordred} | predichos activos: {pred_mordred.sum()} / {len(pred_mordred)}")

# --- Combinar ambos featurizadores por molécula (join por smiles_std) --------
cols_maccs = ["Molecule ChEMBL ID" , "smiles_std", "in_AD_maccs", "ad_dist",
              "maccs_model_top1", "maccs_prob_top1", "maccs_pred_top1", "maccs_top3_votes"]
cols_mordred = ["Molecule ChEMBL ID" , "smiles_std", "in_AD_mordred", "ad_dist",
                "mordred_model_top1", "mordred_prob_top1", "mordred_pred_top1", "mordred_top3_votes"]

df_m1 = df_maccs_cand[cols_maccs].rename(columns={"ad_dist": "ad_dist_maccs"})
df_m2 = df_mordred_cand[cols_mordred].rename(columns={"ad_dist": "ad_dist_mordred"})

df_final = df_m1.merge(df_m2, on=["Molecule ChEMBL ID" , "smiles_std"], how="inner")
print(f"Moléculas con MACCS y Mordred calculados con éxito: {len(df_final)}")

# --- Consenso estricto --------------------------------------------------------
df_final["in_AD_ambos"] = df_final["in_AD_maccs"] & df_final["in_AD_mordred"]
df_final["activo_ambos_top1"] = (df_final["maccs_pred_top1"] == 1) & (df_final["mordred_pred_top1"] == 1)
df_final["candidato_MoA_FP2"] = df_final["in_AD_ambos"] & df_final["activo_ambos_top1"]

df_final["prob_media"] = (df_final["maccs_prob_top1"] + df_final["mordred_prob_top1"]) / 2

df_final = df_final.sort_values("prob_media", ascending=False).reset_index(drop=True)
df_final.head(20)


MACCS   top-1 = SVM_RBF   | predichos activos: 4345 / 30130
Mordred top-1 = KNeighbors | predichos activos: 4920 / 29752
Moléculas con MACCS y Mordred calculados con éxito: 29752


,Molecule ChEMBL ID,smiles_std,in_AD_maccs,ad_dist_maccs,maccs_model_top1,maccs_prob_top1,maccs_pred_top1,maccs_top3_votes,in_AD_mordred,ad_dist_mordred,mordred_model_top1,mordred_prob_top1,mordred_pred_top1,mordred_top3_votes,in_AD_ambos,activo_ambos_top1,candidato_MoA_FP2,prob_media
0,CHEMBL1793947,CCCC(=O)CCCCC[C@@H]1NC(=O)[C@H]2CCCCN2CC(=O)[C...,True,12.428679,SVM_RBF,0.903761,1,3,True,12.389524,KNeighbors,1.000000,1,1,True,True,True,0.951880
1,CHEMBL3409895,CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)N1CCCCC1)...,True,10.808564,SVM_RBF,0.885271,1,3,True,11.829564,KNeighbors,1.000000,1,2,True,True,True,0.942635
2,CHEMBL363795,COc1ccc(/C=C/C(=O)c2ccc(NC(=O)Nc3ccc(Cl)cc3)cc...,True,7.702257,SVM_RBF,0.872733,1,3,True,7.252021,KNeighbors,1.000000,1,1,True,True,True,0.936366
3,CHEMBL1928842,CCCCOc1c(N[C@@H](CC(C)C)C(=O)NNC(=O)/C=C/S(=O)...,False,16.099251,SVM_RBF,0.866884,1,3,True,13.774791,KNeighbors,1.000000,1,3,False,True,False,0.933442
4,CHEMBL5406969,C[C@H]1C[C@@H](N2C(=N)N[C@](C)(c3cccc(-c4ccc(C...,False,14.014983,SVM_RBF,0.842864,1,2,False,17.377686,KNeighbors,1.000000,1,2,False,True,False,0.921432
5,CHEMBL3133578,COC(=O)[C@H](CCSC)NC(=O)c1sc(SC(C)C)c(C#N)c1-c...,True,12.277369,SVM_RBF,0.827962,1,3,False,16.132035,KNeighbors,1.000000,1,1,False,True,False,0.913981
6,CHEMBL371998,O=C(Nc1ccccc1)Nc1cccc(C(=O)/C=C/c2ccc(Cl)cc2)c1,True,4.007510,SVM_RBF,0.822815,1,3,True,6.063671,KNeighbors,1.000000,1,2,True,True,True,0.911408
7,CHEMBL4435950,CC(C)(C)CC(=O)NC(C(=O)NO)c1ccc(-c2cc(F)c(F)c(F...,False,15.222146,SVM_RBF,0.819213,1,3,False,17.431658,KNeighbors,1.000000,1,3,False,True,False,0.909606
8,CHEMBL3409902,CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)N1CCCCC1)...,True,9.825102,SVM_RBF,0.947626,1,3,True,11.354201,KNeighbors,0.857143,1,2,True,True,True,0.902384
9,CHEMBL3409896,CC(C)C[C@H](NC(=O)CN1CCC(O)CC1)C(=O)N[C@@H](CC...,True,10.777840,SVM_RBF,0.786030,1,3,True,12.278616,KNeighbors,1.000000,1,2,True,True,True,0.893015


In [13]:
n_total       = len(df_candidatos)
n_desc_ok     = len(df_final)
n_in_ad       = df_final["in_AD_ambos"].sum()
n_candidatos  = df_final["candidato_MoA_FP2"].sum()

print("=" * 60)
print("RESUMEN DEL CRIBADO VIRTUAL - hipótesis de MoA: inhibición de FP2")
print("=" * 60)
print(f"Candidatos antiplasmódicos                             : {n_total}")
print(f"  -> con MACCS y Mordred calculados correctamente      : {n_desc_ok}")
print(f"  -> dentro del AD en MACCS y en Mordred               : {n_in_ad} ({n_in_ad/n_desc_ok:.1%})")
print(f"  -> ademas predichos ACTIVOS por ambos modelos (top-1): {n_candidatos} ({n_candidatos/n_desc_ok:.1%})")
print()
print("Estos ultimos son los candidatos cuya posible actividad antiplasmodica")
print("podria explicarse (segun el modelo) por inhibicion de Falcipaina-2.")

out_all  = f"{OUTPUT_DIR_VS}/cribado_FP2_todos_los_candidatos.csv"
out_cand = f"{OUTPUT_DIR_VS}/cribado_FP2_candidatos_MoA.csv"

df_final.to_csv(out_all, index=False)
df_final[df_final["candidato_MoA_FP2"]].to_csv(out_cand, index=False)

print(f"\nGuardado: {out_all}")
print(f"Guardado: {out_cand}")


RESUMEN DEL CRIBADO VIRTUAL - hipótesis de MoA: inhibición de FP2
Candidatos antiplasmódicos                             : 30130
  -> con MACCS y Mordred calculados correctamente      : 29752
  -> dentro del AD en MACCS y en Mordred               : 14225 (47.8%)
  -> ademas predichos ACTIVOS por ambos modelos (top-1): 498 (1.7%)

Estos ultimos son los candidatos cuya posible actividad antiplasmodica
podria explicarse (segun el modelo) por inhibicion de Falcipaina-2.

Guardado: cribado virtual/cribado_FP2_todos_los_candidatos.csv
Guardado: cribado virtual/cribado_FP2_candidatos_MoA.csv
